# nano Beatrix — the pointer screen: one attention head told where the record is

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pointer.ipynb)

**The question**: Alhena (Castor's body turned by the anchored multi-position rotary) answers a question exactly when it is handed the address it asks for, and dozens of independently trained arms then merge under the gate with nothing lost. The address has to come from the model itself, through the index it writes: at each entry, the address of the record that holds the entry's key. With the record's address added to every one of its tokens (Alhena-V), a look that finds the record returns its address, and the copying failure mostly went; but the model's own index still locked on only 2 of 6 seeds. Reading the attention of the runs that stalled: on half of them the head that does the lookup splits its attention between the right record and one whose key ends in the same two characters (a match on part of the key), and on others no head looks back at the records from the index at all. The only signal that trains the lookup today arrives through the two address bytes each index entry writes.

**Alhena-P** tells one head where to look. At each index entry, the attention of head 0 in block 4 (of 6) gets a target: its share of attention on the record whose key the entry carries. The term is the negative log of that share, averaged over the entries,

    L = L_text + L_address_head + lambda * mean over entries of ( - log [ attention mass of block 4, head 0, from the entry, on that record ] )

with no new parameter and the forward pass unchanged. On two records whose keys end the same way the gradient moves the head's attention from the wrong one to the right one directly, and on every seed a head is made to form where the lookup belongs. Everything else is Alhena-V's recipe and read; every run also reads its index on 64 key-hard rows (keys built so that only the whole key finds the record), a read that leaves the training untouched.

| part | what | why |
|---|---|---|
| the dose step | P s1 at lambda .1, then at .3 | the rule written before any run picks the dose for the other seeds: the smaller dose whose head reaches .9 of its attention on the record by step 4,000 with the held-out text within .0209 bits of Alhena-V's seed 1 |
| P s2-s6 | Alhena-V + the pointer term at the chosen dose | does the model's own lookup (its written index) lock on at least 5 of 6 seeds |
| V s7, s8 | Alhena-V again on two new seeds (series `valuerep`) | the replication arm: the base's lock rate over runs |
| members | the scale test's 48 real arms + 48 toy members (after the last block) and 16 real arms (after block 3), each trained alone, on the first two P cores (in seed order, at the chosen dose) that lock | the request = the model's own written index (free-running); merged under w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)) |

**The cells**: C1 pins the trainer and the arm library and holds the decision block and the dose rule; C2 builds the data, runs 200 steps of Alhena-P on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 trains P s1 at both doses and applies the rule; C4 trains the members on P s1 at the chosen dose if its written index is right on at least .98 of its entries (free-running) and its answers with the true address reach .95; C5 trains P s2, P s3, V s7, P s4, P s5, V s8 and P s6; C6 trains the members on the next P cores in seed order while fewer than two cores carry them; C7 renders the tables. Every finished run ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/` with the address tables in `value_addr.safetensors`, `bed/scale_members/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/pointer.py`, `bed_train.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK + THE DOSE RULE. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.78" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 78), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the members, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000,                     # the formation trace every 2,000 steps (the pointer's mass read at each point)
                 hard_read=1)                          # the index on 64 key-hard rows at the end (a read: the training unchanged)
WIDE = dict(rotary_content=4)                          # ALHENA-W: 4 position / 4 content / 8 address pairs
VALUE = dict(value_addr=1)                             # ALHENA-V: the value-path tables, born zero (the name carries _va1)
POINTER = dict(ptr_block=4, ptr_head=0, series='pointer')   # ALHENA-P: block 4 of 6, head 0 (the name carries _pt<100 lambda>b4h0)
DOSES = (0.1, 0.3)                                     # the dose step: P s1 at each; the rule below picks the dose for seeds 2-6
P = lambda s, lam: (f'P s{s} lambda {lam:g}', dict(BED, **GRID_ROWS, **WIDE, **VALUE, **POINTER, ptr_lam=lam, arms='alhena-both', seeds=str(s)))
V = {s: (f'V s{s}', dict(BED, **GRID_ROWS, **WIDE, **VALUE, arms='alhena-both', seeds=str(s), series='valuerep')) for s in (7, 8)}
DOSE_STEP = [P(1, lam) for lam in DOSES]               # C3: the dose step
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
TAG_P = lambda s, lam: f'alhena-both_s{s}{TAIL}_cp4_va1_pt{int(round(lam * 100))}b4h0_ahdigit1_pointer'
V1_BITS, BITS_MARGIN = 1.4264, 0.0209                  # Alhena-V seed 1's held-out bits (the value-path screen) and the one-sided text bar
MASS_BAR, BY_STEP = 0.9, 4000                          # the head's mass on the record by step 4,000 (the plan's P1)
MIN_INDEX, MIN_ANSWERS = 0.98, 0.95                    # the plan's member rule: the written index (free-running) >= .98, answers >= .95
N_MEMBER_CORES = 2                                     # the first two P cores at the chosen dose, in seed order, that pass the rule
LAST_STEPS = 16000                                     # the scale test's member budget, verbatim
MID_STEPS = 32000                                      # batch steps at the mid site (16,000 own steps a member)
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the scale test's read and calibration rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of Alhena-P on this card for the price line; its record stays local
PRICE_STEPS = 200                                      # the member price lines (each trainer on throwaway stacks)
PLAN_MIN = 16.6                                        # the planned minutes of one 24,000-step core on this card (16.1-16.3 measured on Alhena-V; the term est. < 1%)
PLAN_MIN_MEM = 18.8                                    # the planned training minutes of one core's members (4.9 last + 13.8 mid measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card

# ============ THE DOSE RULE (the plan's section 4, written before any run; ptr_dose.dose_rule verbatim) ============
def dose_rule(rec_lo, rec_hi, lam_lo=0.1, lam_hi=0.3, v1_bits=1.4264, bits_margin=0.0209, mass_bar=0.9, by_step=4000):
    """The dose for seeds 2-6 from the two seed-1 records: the smaller dose if its supervised head reaches mass >= mass_bar on the trace
    rows by step `by_step` AND its held-out bits <= V s1's + bits_margin; else the larger if it meets both; else the dose whose free-running
    written index is higher (a tie: the smaller). -> (lambda*, {dose: (mass by the step, bits, the free-running index, meets both)})."""
    def marks(r):
        tr = [t for t in r["grid"]["trace"] if 0 < t["step"] <= by_step and t.get("ptr_mass") is not None]
        m = max((t["ptr_mass"] for t in tr), default=0.0)
        bits = r["heldout_bpb"]
        fr = (r["grid"]["read"].get("free_running") or {}).get("indexf_right", 0.0)
        return m, bits, fr, (m >= mass_bar and bits <= round(v1_bits + bits_margin, 10))     # the bar as written (1.4473), not its float sum
    lo, hi = marks(rec_lo), marks(rec_hi)
    if lo[3]:
        lam = lam_lo
    elif hi[3]:
        lam = lam_hi
    else:
        lam = lam_lo if lo[2] >= hi[2] else lam_hi
    return lam, {lam_lo: lo, lam_hi: hi}

print('decision block: the dose step', [n for n, _ in DOSE_STEP], '+ 7 cores | the member rule', MIN_INDEX, MIN_ANSWERS, '| out', OUT,
      '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena-P on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo = 0
for name, spec in DOSE_STEP + [P(s, DOSES[0]) for s in (2, 3, 4, 5, 6)] + [V[7], V[8]]:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']
smoke = dict(DOSE_STEP[0][1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2, hard_batches=1)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_run <= STOP_FACTOR * PLAN_MIN
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.1f} min (planned "
      f"{PLAN_MIN} min); the {todo} runs left about {per_run * todo / 60:.1f} h (+ a few minutes of end reads a run, + the members)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_pointer.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [n for n, _ in DOSE_STEP] + ['P s2-s6 at the chosen dose', 'V s7', 'V s8']}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano pointer: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

In [ ]:
# C3 (T) — THE DOSE STEP: P s1 at lambda .1, then at .3, one run at a time, resume-first (runner_done.txt in OUT); a STOP file in OUT
# starts no new run; every finished run ships to REPO as it lands (one background shipper: the card never waits on the Hub). Then THE
# DOSE RULE on the two records: LAM, the dose of seeds 2-6.
PARTS_NOW = DOSE_STEP
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])
_r = {lam: json.load(open(f'{OUT}/bed/bed_{TAG_P(1, lam)}.json', encoding='utf-8')) for lam in DOSES}
LAM, _marks = dose_rule(_r[DOSES[0]], _r[DOSES[1]], DOSES[0], DOSES[1], V1_BITS, BITS_MARGIN, MASS_BAR, BY_STEP)
for lam, (m, b, fr, ok) in _marks.items():
    print(f'P s1 at lambda {lam:g}: the head\'s mass on the record by step {BY_STEP:,} {m:.4f} (bar {MASS_BAR}), held-out bits {b:.4f} '
          f'(bar {V1_BITS + BITS_MARGIN:.4f}), the written index free-running {fr:.4f} -> meets both: {ok}')
print(f'THE DOSE RULE -> lambda {LAM:g} for P s2-s6; P s1 at lambda {LAM:g} is the screen\'s seed 1')

In [ ]:
# C4 (T) — THE MEMBERS on P s1 at the chosen dose: if its own end read writes its index right on >= .98 of the entries (free-running)
# and answers >= .95 with the true address, it carries the scale test's members verbatim; the request is the core's own written index
# (the two-pass read, the question-head set: LAYOUT, INDEX, INDEXC, INDEXF, HEADPF, HEADPFQ999, HEADI, HEADIF, G2, G0). Fewer than two
# member cores -> C6 takes the next P cores in seed order.
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr.get('indexf_right', 0.0)
    ok = look >= MIN_INDEX and ans >= MIN_ANSWERS
    return ok, f"its written index (free-running) {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    print('the core\'s address form:', 'one byte' if core.abytes == 1 else 'two digits')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
MEMBER_CORES = []
ok, why = _qualifies(TAG_P(1, LAM))
print(f'P s1 at lambda {LAM:g}: {why} -> ' + ('its members train' if ok else 'NOT HERE (C6 takes the next P core that passes)'))
if ok:
    MEMBER_CORES.append((TAG_P(1, LAM), 1))
done_m = run_members(MEMBER_CORES) if MEMBER_CORES else []
print('members:', [r['name'] for r in done_m], '| member cores still wanted:', N_MEMBER_CORES - len(MEMBER_CORES))

In [ ]:
# C5 (T) — THE OTHER CORES at the chosen dose: P s2, P s3, then the replication arm between the P seeds (V s7, P s4, P s5, V s8, P s6);
# one run at a time, resume-first, shipped as each lands.
PARTS_NOW = [P(2, LAM), P(3, LAM), V[7], P(4, LAM), P(5, LAM), V[8], P(6, LAM)]
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS_NOW:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C6 (T) — THE SUBSTITUTION RULE (the plan's, written before any run): while fewer than two P cores carry members, the next P core at the
# chosen dose in seed order (P s2, then P s3 ... P s6) that passes the same rule carries them.
import json, time
from geolip.nano_beatrix import rotary_scale as rs
BED_DIR = f'{OUT}/bed'

def _record(tag):
    p = f'{BED_DIR}/bed_{tag}.json'
    return json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None

def _qualifies(tag):
    r = _record(tag)
    if r is None:
        return None, 'no record in this session'
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    ans = g['true']['whole']
    look = fr.get('indexf_right', 0.0)
    ok = look >= MIN_INDEX and ans >= MIN_ANSWERS
    return ok, f"its written index (free-running) {look:.4f}, answers with the true address {ans:.4f}"

def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(BED_DIR)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{BED_DIR}/{cores[0][0]}', 'cuda')
    print('the core\'s address form:', 'one byte' if core.abytes == 1 else 'two digits')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {tag} ===================')
        t0 = time.time()
        res = rs.run_core(f'{BED_DIR}/{tag}', f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out
want = N_MEMBER_CORES - len(globals().get('MEMBER_CORES', []))
SUB = []
for s in (2, 3, 4, 5, 6):
    if len(SUB) >= want:
        break
    ok, why = _qualifies(TAG_P(s, LAM))
    print(f'P s{s}: {why} -> ' + ('its members train' if ok else 'not a member core'))
    if ok:
        SUB.append((TAG_P(s, LAM), s))
done_s = run_members(SUB) if SUB else []
print('substitute members:', [r['name'] for r in done_s] or ('none needed' if want <= 0 else 'none (no other P core passed)'))

In [ ]:
# C7 — THE TABLES: the cores (bits, the frame copy, the index both ways, the key-hard rows' index both ways, the answers, the pointer's
# dose and its head's mass at steps 2,000 / 4,000 / the end read, minutes) and the members (per core and stack: solo against bare, the
# nine rest cells keeping all three marks per source, the controls, the largest leak of the written index, the mid site, the text bits).
# Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | key-hard rows TF / FR | answers true / via index | pointer: lambda, mass 2k / 4k / read, descended | min |',
        '|---|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_alhena-both_*_L24000_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    if not r.get('grid'):
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hd = g.get('hard')
    hard = f"{hd['index_right']:.4f} / {hd['indexf_right']:.4f}" if hd else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f}"
    pt_ = r['grid'].get('ptr')
    if pt_:
        tm = {t['step']: t.get('ptr_mass') for t in r['grid']['trace']}
        ptr = (f"{pt_['lam']:g}, {tm.get(2000, float('nan')):.3f} / {tm.get(4000, float('nan')):.3f} / {pt_['mass_read']:.3f}, "
               f"{pt_['descended']}")
    else:
        ptr = '-'
    _n = os.path.basename(p)[4:-5]
    rows.append(f"| {_n.split('_ctx')[0]} {_n.split('gag50_')[-1]} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / "
                f"{fr.get('indexf_right', float('nan')):.4f} | {hard} | {via} | {ptr} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| core | stack | solo / bare owned | INDEXF | HEADPF | LAYOUT | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (INDEXF) | mid INDEXF U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    if not os.path.exists(f'{d}/last_find.json'):
        continue
    L = json.load(open(f'{d}/last_find.json', encoding='utf-8'))
    M = json.load(open(f'{d}/mid_find.json', encoding='utf-8')) if os.path.exists(f'{d}/mid_find.json') else None
    rd = L['read']
    for st, r in rd['stacks'].items():
        c = r['cells']
        allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
        nine = lambda s: f"{sum(allm(c[f'{s}_U{u}_REST_m{m}']) for u in (5, 8, 10) for m in (16, 32, 48))}/9" if f'{s}_U5_REST_m16' in c else '-'
        leak = max(c[f'INDEXF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'INDEXF_U{u}_REST_m16'])) if f'INDEXF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = L['text_bits'].get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:30]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | {nine('INDEXF')} | {nine('HEADPF')} | "
                    f"{nine('LAYOUT')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | {g0:+.3f} |")
display(Markdown('\n'.join(rows)))